# 读写文件

到目前为止，我们讨论了如何处理数据，
以及如何构建、训练和测试深度学习模型。
然而，有时我们希望保存训练的模型，
以备将来在各种环境中使用（比如在部署中进行预测）。
此外，当运行一个耗时较长的训练过程时，
最佳的做法是定期保存中间结果，
以确保在服务器电源被不小心断掉时，我们不会损失几天的计算结果。
因此，现在是时候学习如何加载和存储权重向量和整个模型了。

## (**加载和保存张量**)

对于单个张量，我们可以直接调用`load`和`save`函数分别读写它们。
这两个函数都要求我们提供一个名称，`save`要求将要保存的变量作为输入。


In [3]:
import torch
from torch import nn
from torch.nn import functional as F

# 创建包含0、1、2、3的一维张量
x = torch.arange(4)

# 将张量x序列化并保存到本地文件x-file中
torch.save(x, 'x-file')

我们现在可以将存储在文件中的数据读回内存。


In [5]:
# 安全加载之前保存的张量
x2 = torch.load('x-file', weights_only=True)

# 查看加载后的张量
print(x2)

tensor([0, 1, 2, 3])


我们可以**存储一个张量列表，然后把它们读回内存。**


In [6]:
# 创建长度为4、所有元素均为0的张量
y = torch.zeros(4)

# 将两个张量组成列表，并保存到文件中
torch.save([x, y], 'x-files')

# 从文件中加载两个张量，分别赋值给x2和y2
x2, y2 = torch.load('x-files', weights_only=True)

# 查看加载后的两个张量
(x2, y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

我们甚至可以(**写入或读取从字符串映射到张量的字典**)。
当我们要读取或写入模型中的所有权重时，这很方便。


In [7]:
# 创建字典，分别使用'x'和'y'作为键，存储对应的张量
mydict = {'x': x, 'y': y}

# 将整个字典保存到本地文件
torch.save(mydict, 'mydict')

# 从文件中加载字典
mydict2 = torch.load('mydict', weights_only=True)

# 查看恢复后的字典
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

## **加载和保存模型参数**

保存单个权重向量（或其他张量）确实有用，
但是如果我们想保存整个模型，并在以后加载它们，
单独保存每个向量则会变得很麻烦。
毕竟，我们可能有数百个参数散布在各处。
因此，深度学习框架提供了内置函数来保存和加载整个网络。
需要注意的一个重要细节是，这将保存模型的参数而不是保存整个模型。
例如，如果我们有一个3层多层感知机，我们需要单独指定架构。
因为模型本身可以包含任意代码，所以模型本身难以序列化。
因此，为了恢复模型，我们需要用代码生成架构，
然后从磁盘加载参数。
让我们从熟悉的多层感知机开始尝试一下。


In [8]:
# 自定义多层感知机（MLP）
class MLP(nn.Module):
    def __init__(self):
        super().__init__()  # 初始化父类

        # 定义隐藏层：输入20维，输出256维
        self.hidden = nn.Linear(20, 256)

        # 定义输出层：输入256维，输出10维
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        # 输入经过隐藏层和ReLU激活，再经过输出层
        return self.output(F.relu(self.hidden(x)))


net = MLP()                      # 实例化MLP模型
X = torch.randn(size=(2, 20))   # 生成2个样本，每个样本20个特征
Y = net(X)                      # 执行前向传播，得到预测结果

接下来，我们**将模型的参数存储在一个叫做“mlp.params”的文件中。**


In [9]:
torch.save(net.state_dict(), 'mlp.params')

为了恢复模型，我们[**实例化了原始多层感知机模型的一个备份。**]
这里我们不需要随机初始化模型参数，而是(**直接读取文件中存储的参数。**)


In [11]:
# 保存模型的所有参数（权重和偏置）
torch.save(net.state_dict(), 'mlp.params')

# 创建一个结构相同的新模型
clone = MLP()

# 从文件中加载参数，并赋值给新模型
clone.load_state_dict(
    torch.load('mlp.params', weights_only=True)
)

# 将模型切换为评估模式
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

由于两个实例具有相同的模型参数，在输入相同的`X`时，
两个实例的计算结果应该相同。
让我们来验证一下。


In [12]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

## 小结

* `save`和`load`函数可用于张量对象的文件读写。
* 我们可以通过参数字典保存和加载网络的全部参数。
* 保存架构必须在代码中完成，而不是在参数中完成。